# 01 - Exploratory Data Analysis: Drug Review (Drugs.com)

**Task:** 3-class sentiment classification of patient reviews (negative / neutral / positive, derived from the 1-10 rating).

This notebook looks at the raw data, the class imbalance, review lengths, vocabulary coverage and a few data-quality issues. It uses the same cleaning and the same fixed split as the models (`src/data.py`), and saves its figures to `results/figures/`.

## 0. Setup
The next cell finds the project code (local folder, Kaggle dataset, or GitHub clone).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_NAME = "projet-lstm-drug-review"
REPO_URL = "https://github.com/YOUR_USERNAME/projet-lstm-drug-review.git"  # <-- edit after publishing

ON_KAGGLE = Path("/kaggle/input").exists()


def locate_repo() -> Path:
    """Find the project code: current folder, uploaded Kaggle dataset, or GitHub clone."""
    # 1) already inside the repository (local machine, or already cloned)
    for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / REPO_NAME):
        if (folder / "src" / "train.py").is_file():
            return folder
    # 2) code uploaded as a Kaggle dataset (folder containing src/train.py)
    if ON_KAGGLE:
        for train_py in Path("/kaggle/input").rglob("train.py"):
            if train_py.parent.name == "src":
                dest = Path("/kaggle/working") / REPO_NAME
                if not dest.exists():
                    shutil.copytree(train_py.parent.parent, dest)
                return dest
    # 3) clone from GitHub (needs Internet enabled in the Kaggle notebook settings)
    dest = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()) / REPO_NAME
    if not dest.exists():
        subprocess.run(["git", "clone", REPO_URL, str(dest)], check=True)
    return dest


REPO = locate_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from src.data import (DataConfig, load_raw, build_bundle_from_dataframes, clean_text,
                      tokenize, compute_class_weights)

pd.set_option("display.max_colwidth", 110)
FIG_DIR = REPO / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Rating -> class mapping is configurable: edges (4, 6) means negative 1-4, neutral 5-6, positive 7-10.
CFG = DataConfig(label_edges=(4, 6), max_len=200, min_freq=2, max_vocab=30000, split_seed=42)

## 1. Raw data overview

In [ ]:
train_raw, test_raw = load_raw(CFG.data_dir)
print("official train:", train_raw.shape, "| official test:", test_raw.shape)
display(train_raw.head(3))
display(train_raw.dtypes.to_frame("dtype"))

In [ ]:
print("Missing values")
display(pd.DataFrame({"train": train_raw.isna().sum(), "test": test_raw.isna().sum()}))
print("duplicated uniqueID (train):", train_raw["uniqueID"].duplicated().sum())
print("duplicated review text (train):", train_raw["review"].duplicated().sum())

## 2. Cleaning and the fixed train / validation / test split
HTML entities are decoded, URLs removed, text lower-cased. The validation set is a stratified 10% of the official training file; the test set is the official test file; the vocabulary uses the training split only.

In [ ]:
bundle = build_bundle_from_dataframes(train_raw, test_raw, CFG)
print(bundle.summary())

has_entity = train_raw["review"].astype(str).str.contains(r"&#?\w+;", regex=True)
print(f"\nReviews containing an HTML entity (before cleaning): {has_entity.mean():.1%}")
if has_entity.any():
    example = train_raw.loc[has_entity, "review"].iloc[0]
    print("BEFORE:", example[:250])
    print("AFTER :", clean_text(example)[:250])

## 3. Ratings and class imbalance

In [ ]:
rating_counts = train_raw["rating"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rating_counts.index.astype(int), rating_counts.values, color="tab:blue")
ax.set(title="Rating distribution (official train file)", xlabel="Rating", ylabel="Number of reviews")
ax.set_xticks(rating_counts.index.astype(int))
ax.grid(axis="y", alpha=0.3)
fig.savefig(FIG_DIR / "eda_rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rows = {}
for split in ("train", "val", "test"):
    counts = np.bincount(bundle.y[split], minlength=bundle.num_classes)
    rows[split] = pd.Series(counts, index=bundle.class_names)
class_counts = pd.DataFrame(rows)
class_pct = (class_counts / class_counts.sum() * 100).round(1).add_suffix(" (%)")
display(pd.concat([class_counts, class_pct], axis=1))

train_counts = class_counts["train"]
print(f"Imbalance ratio (largest / smallest class, train): {train_counts.max() / train_counts.min():.2f}")
weights = compute_class_weights(bundle.y["train"], bundle.num_classes)
print("Inverse-frequency class weights:", {n: round(float(w), 3) for n, w in zip(bundle.class_names, weights)})

fig, ax = plt.subplots(figsize=(7, 4))
share = class_counts / class_counts.sum()
share.T.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set(title="Class proportions per split", ylabel="Share of reviews", xlabel="")
ax.legend(title="Class")
plt.xticks(rotation=0)
fig.savefig(FIG_DIR / "eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading guide (fill in after looking at the numbers):** how strong is the imbalance, which class is the minority, and what does that imply for accuracy vs macro-F1 and for class weights? Is the test distribution close to the training one?

## 4. Review length and truncation

In [ ]:
train_df = bundle.splits["train"].copy()
train_df["n_words"] = train_df["review_clean"].str.split().str.len()
train_df["class_name"] = train_df["label"].map(dict(enumerate(bundle.class_names)))

print("Review length in words (train):")
display(train_df["n_words"].describe().round(1).to_frame("all reviews").T)
display(train_df.groupby("class_name")["n_words"].describe().round(1))

print(f"\nShare of reviews longer than max_len = {CFG.max_len}: {(train_df['n_words'] > CFG.max_len).mean():.1%}")
for length in (50, 100, 150, 200, 300):
    print(f"  longer than {length:>3} words: {(train_df['n_words'] > length).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_df["n_words"].clip(upper=400), bins=40, color="tab:orange")
ax.axvline(CFG.max_len, color="red", linestyle="--", label=f"max_len = {CFG.max_len}")
ax.set(title="Review length (words, clipped at 400)", xlabel="Words", ylabel="Number of reviews")
ax.legend()
fig.savefig(FIG_DIR / "eda_review_length.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Conditions, drugs, helpfulness, dates

In [ ]:
top_conditions = train_raw["condition"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(8, 5))
top_conditions[::-1].plot(kind="barh", ax=ax, color="tab:green")
ax.set(title="Top 15 conditions (train)", xlabel="Number of reviews")
fig.savefig(FIG_DIR / "eda_top_conditions.png", dpi=150, bbox_inches="tight")
plt.show()

print("Number of distinct conditions:", train_raw["condition"].nunique())
print("Missing condition:", train_raw["condition"].isna().sum())
junk = train_raw["condition"].astype(str).str.contains("</span>", regex=False).sum()
print("Conditions polluted by HTML ('</span> users found this comment helpful'):", junk)
print("\nTop 10 drugs:")
display(train_raw["drugName"].value_counts().head(10).to_frame("reviews"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log1p(train_raw["usefulCount"]), bins=40, color="tab:purple")
axes[0].set(title="usefulCount (log1p)", xlabel="log(1 + usefulCount)", ylabel="Reviews")
years = pd.to_datetime(train_raw["date"], errors="coerce").dt.year.value_counts().sort_index()
axes[1].bar(years.index.astype(int), years.values, color="tab:red")
axes[1].set(title="Reviews per year", xlabel="Year", ylabel="Reviews")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_useful_and_dates.png", dpi=150, bbox_inches="tight")
plt.show()
print(train_raw["usefulCount"].describe().round(1).to_frame("usefulCount").T)

## 6. Vocabulary coverage and class-specific words
The vocabulary is built from the training split only. Out-of-vocabulary (OOV) rates on validation and test tell us how much information `<UNK>` hides.

In [ ]:
vocab = bundle.vocab
print(f"Vocabulary size (incl. <PAD>, <UNK>): {len(vocab):,}  (min_freq={CFG.min_freq}, max_vocab={CFG.max_vocab:,})")

def oov_rate(split):
    tokens = [tok for text in bundle.splits[split]["review_clean"] for tok in tokenize(text)]
    return float(np.mean([tok not in vocab.stoi for tok in tokens]))

for split in ("train", "val", "test"):
    print(f"OOV token rate on {split}: {oov_rate(split):.2%}")

In [ ]:
MIN_COUNT = 50
per_class = {k: Counter() for k in range(bundle.num_classes)}
for text, label in zip(train_df["review_clean"], train_df["label"]):
    per_class[label].update(tokenize(text))
totals = {k: sum(c.values()) for k, c in per_class.items()}
all_words = set().union(*[set(c) for c in per_class.values()])

def log_odds(word, k):
    in_k = per_class[k][word]
    rest = sum(per_class[j][word] for j in per_class if j != k)
    n_rest = sum(totals[j] for j in per_class if j != k)
    return np.log((in_k + 1) / totals[k]) - np.log((rest + 1) / n_rest)

table = {}
for k, name in enumerate(bundle.class_names):
    candidates = [w for w in all_words if sum(per_class[j][w] for j in per_class) >= MIN_COUNT]
    ranked = sorted(candidates, key=lambda w: log_odds(w, k), reverse=True)[:12]
    table[name] = ranked
display(pd.DataFrame(table))
print("Words most over-represented in each class (log-odds vs the other classes, min count", MIN_COUNT, ")")

## 7. Train / test overlap (leakage check)

In [ ]:
overlap = set(bundle.splits["train"]["review_clean"]) & set(bundle.splits["test"]["review_clean"])
print("Identical cleaned reviews present in both train and test:", len(overlap))
overlap_val = set(bundle.splits["train"]["review_clean"]) & set(bundle.splits["val"]["review_clean"])
print("Identical cleaned reviews present in both train and validation:", len(overlap_val))

## 8. Summary for the report
Write 5-8 bullet points here **after running the notebook** (use the real numbers):
- class distribution and imbalance ratio -> why macro-F1 + optional class weights;
- review length and the choice of `max_len`;
- vocabulary size, OOV rate;
- data-quality issues (HTML entities, polluted `condition` values, duplicates);
- anything that could explain model errors (e.g. confusion between neutral and the other classes).